In [4]:
import os
import pandas as pd
import re
import time
import pdfplumber
from datetime import datetime
from collections import defaultdict

# ---------------------------
# CONFIGURATION
# ---------------------------

start = time.time()

# Update this to your actual path for PriBank
BASE_DIR = "/Users/test/Desktop/Stuff/pri_financial_data"
CURRENT_DIR = os.path.join(BASE_DIR, "current")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

for subfolder in ['balance-sheet', 'income-statement']:
    os.makedirs(os.path.join(CURRENT_DIR, subfolder), exist_ok=True)
    print(f"📁 Directory ready: {os.path.join(CURRENT_DIR, subfolder)}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Output directory ready: {OUTPUT_DIR}")

# ---------------------------
# DATA STORAGE
# ---------------------------

extracted_data = defaultdict(list)

# ---------------------------
# FUNCTIONS
# ---------------------------

def extract_date(text, filename):
    """Extract a date string from text or filename"""
    match = re.search(r'\d{1,2}[.]\d{1,2}[.]\d{4}', text)
    if match:
        parts = [p.zfill(2) for p in match.group().split('.')]
        return '.'.join(parts)
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', text)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', filename)
    if match:
        return match.group().replace('-', '.').replace('/', '.')
    return None

def replace_negatives(value):
    """Convert parenthesized numbers to negative"""
    if pd.isna(value) or str(value).strip() in ['', '-', '0']:
        return '0'
    val = str(value).strip()
    if '(' in val and ')' in val:
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

# Define financial categories for PriBank
income_statement_categories = [
    'Të hyrat nga interesi / Interest income',
    'Shpenzimet e interesit / Interest expense',
    'Neto të hyrat nga interesi / Net interest income',
    'Fee and commission income',
    'Fee and commission expense',
    'Net fee and commission income',
    'Net trading profit',
    'Net income from other financial instruments',
    'Net other operating income (expense)',
    'Gjithsej të hyrat/Total income',
    'Impairment losses on financial assets',
    'Fitimi/(humbja) para tatimit / Profit/(loss) before taxation',
    'Fitimi/(humbja) neto / Net profit/(loss)',
    'Other comprehensive income',
    'Total comprehensive income/(loss)'
]

balance_sheet_categories = [
    'Paraja e gatshme dhe gjendja me Bankat Qendrore',
    'Kërkesat ndaj bankave',
    'Investimet në letra me vlerë',
    'Kreditë dhe paradhëniet ndaj bankave',
    'Kreditë dhe paradhëniet ndaj klientëve',
    'Patundshmëritë dhe pajisjet',
    'Pasuritë e paprekshme',
    'Pasuritë tatimore të shtyra',
    'Pasuritë tjera',
    'GJITHSEJ PASURITË',
    'Customer Deposits',
    'Other liabilities',
    'Gjithsej detyrimet',
    'Kapitali aksionar',
    'Rezervat e kapitalit',
    'Fitimi i mbajtur/(humbja) nga vitet paraprake',
    'Fitimi/(humbja) e vitit aktual',
    'Përbërësit tjerë të ekuitetit',
    'Gjithsej ekuiteti i aksionarëve',
    'GJITHSEJ DETYRIMET DHE EKUITETI I AKSIONARËVE'
]

target_categories = {
    "Income Statement": income_statement_categories,
    "Balance Sheet": balance_sheet_categories,
}

def process_pdf(pdf_path, filename):
    """Process a single PDF and extract financial data"""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if not text:
                    continue
                dt_report = extract_date(text, pdf_path)
                if dt_report:
                    date_obj = datetime.strptime(dt_report, '%d.%m.%Y')
                    dt_report_formatted = date_obj.strftime('%Y-%m-%d')
                else:
                    dt_report_formatted = datetime.now().strftime('%Y-%m-%d')
                
                for doc_type, categories in target_categories.items():
                    categories_processed = set()
                    for line in text.split('\n'):
                        for category in categories:
                            if category.lower() in line.lower() and category not in categories_processed:
                                categories_processed.add(category)
                                start_index = line.find(category) + len(category)
                                line_rest = line[start_index:]
                                values = re.findall(r'\(?\d+[,.]?\d*\)?', line_rest)
                                if values and len(values) >= 2:
                                    extracted_data[doc_type].append({
                                        'Category': category,
                                        'PREVIOUS_QUARTER': values[0],
                                        'CURRENT_QUARTER': values[1],
                                        'DT_REPORT': dt_report_formatted,
                                        'FILE_NAME': filename
                                    })
        print(f"   ✅ Processed: {filename}")
    except Exception as e:
        print(f"   ❌ Error processing {filename}: {e}")

def clean_numeric_values(df, prev_col, curr_col):
    for col in [prev_col, curr_col]:
        df[col] = df[col].astype(str).str.replace('-', '0', regex=False)
        df[col] = df[col].apply(replace_negatives)
        df[col] = df[col].str.replace(',', '')
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

# Category mapping for PriBank
def map_pri_categories(df, is_balance=True):
    """Map categories to standardized codes"""
    bs_map = {
        'Paraja e gatshme dhe gjendja me Bankat Qendrore': '20',
        'Kërkesat ndaj bankave': '21',
        'Investimet në letra me vlerë': '23',
        'Kreditë dhe paradhëniet ndaj bankave': '25',
        'Kreditë dhe paradhëniet ndaj klientëve': '26',
        'Patundshmëritë dhe pajisjet': '27',
        'Pasuritë e paprekshme': '28',
        'Pasuritë tatimore të shtyra': '29',
        'Pasuritë tjera': '30',
        'GJITHSEJ PASURITË': '31',
        'Customer Deposits': '32',
        'Other liabilities': '36',
        'Gjithsej detyrimet': '37',
        'Kapitali aksionar': '38',
        'Rezervat e kapitalit': '40',
        'Fitimi i mbajtur/(humbja) nga vitet paraprake': '41',
        'Fitimi/(humbja) e vitit aktual': '42',
        'Përbërësit tjerë të ekuitetit': '43',
        'Gjithsej ekuiteti i aksionarëve': '44',
        'GJITHSEJ DETYRIMET DHE EKUITETI I AKSIONARËVE': '45'
    }
    
    is_map = {
        'Të hyrat nga interesi / Interest income': '1',
        'Shpenzimet e interesit / Interest expense': '2',
        'Neto të hyrat nga interesi / Net interest income': '3',
        'Fee and commission income': '4',
        'Fee and commission expense': '5',
        'Net fee and commission income': '6',
        'Net trading profit': '7',
        'Net income from other financial instruments': '8',
        'Net other operating income (expense)': '9',
        'Gjithsej të hyrat/Total income': '10',
        'Impairment losses on financial assets': '11',
        'Fitimi/(humbja) para tatimit / Profit/(loss) before taxation': '14',
        'Fitimi/(humbja) neto / Net profit/(loss)': '16',
        'Other comprehensive income': '17',
        'Total comprehensive income/(loss)': '19'
    }
    
    if is_balance:
        df['CATEGORY_CODE'] = df['BALANCE_CATEGORY'].map(bs_map)
        df['STATEMENT_TYPE'] = 'BALANCE_SHEET'
    else:
        df['CATEGORY_CODE'] = df['INCOME_CATEGORY'].map(is_map)
        df['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
    
    df['BANK_ID'] = 12
    df['CURR_ID'] = 1
    return df

# ---------------------------
# MAIN EXECUTION
# ---------------------------

def main():
    global extracted_data
    extracted_data.clear()
    
    # Collect PDF files from both folders
    pdf_files = []
    for category in ['balance-sheet', 'income-statement']:
        category_dir = os.path.join(CURRENT_DIR, category)
        if os.path.exists(category_dir):
            pdf_files.extend([os.path.join(category_dir, f) for f in os.listdir(category_dir)
                              if f.lower().endswith('.pdf')])
    
    print(f"📂 Found {len(pdf_files)} PDF files for processing")
    
    # Process PDFs
    for pdf_path in pdf_files:
        process_pdf(pdf_path, os.path.basename(pdf_path))
    
    # Build DataFrames
    full_bs = pd.DataFrame(extracted_data['Balance Sheet'])
    full_is = pd.DataFrame(extracted_data['Income Statement'])
    
    if not full_bs.empty:
        full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
        full_bs = clean_numeric_values(full_bs, 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE')
        full_bs = map_pri_categories(full_bs, is_balance=True)
        full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'])
    
    if not full_is.empty:
        full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
        full_is = clean_numeric_values(full_is, 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE')
        full_is = map_pri_categories(full_is, is_balance=False)
        full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'])
    
    # Save outputs
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    if not full_is.empty:
        full_is.to_csv(os.path.join(OUTPUT_DIR, f"pri_income_statement_{timestamp}.csv"), index=False)
    if not full_bs.empty:
        full_bs.to_csv(os.path.join(OUTPUT_DIR, f"pri_balance_sheet_{timestamp}.csv"), index=False)
    
    # Create unified Excel
    if not full_is.empty or not full_bs.empty:
        excel_path = os.path.join(OUTPUT_DIR, f"pri_financial_report_{timestamp}.xlsx")
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            if not full_is.empty:
                full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
            if not full_bs.empty:
                full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
        print(f"💾 Excel report saved to: {excel_path}")
    
    # Create unified final DataFrame with both statements
    unified_dfs = []
    if not full_is.empty:
        is_unified = full_is.rename(columns={'INCOME_CATEGORY': 'ORIGINAL_CATEGORY', 
                                             'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE',
                                             'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'})
        is_unified['CATEGORY_TYPE'] = 'INCOME'
        unified_dfs.append(is_unified)
    
    if not full_bs.empty:
        bs_unified = full_bs.rename(columns={'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY',
                                             'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE',
                                             'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'})
        bs_unified['CATEGORY_TYPE'] = 'BALANCE'
        unified_dfs.append(bs_unified)
    
    if unified_dfs:
        final_df = pd.concat(unified_dfs, ignore_index=True)
        final_df['DATA_SOURCE'] = 'PriBank'
        final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
        final_df['REPORT_DATE'] = final_df['DT_REPORT'].dt.strftime('%Y-%m-%d')
        final_df["REPORT_DATE"] = final_df["REPORT_DATE"].fillna(datetime.now().strftime('%Y-%m-%d'))
        
        # Save unified DataFrame
        unified_path = os.path.join(OUTPUT_DIR, f"pri_unified_financial_data_{timestamp}.csv")
        final_df.to_csv(unified_path, index=False)
        print(f"💾 Unified data saved to: {unified_path}")
    
    print("\n📊 Data extraction completed")
    print(f"Income Statement rows: {len(full_is)}")
    print(f"Balance Sheet rows: {len(full_bs)}")
    
    elapsed_time = time.time() - start
    print(f"\n⏱️ Completed in {elapsed_time:.2f} seconds")
    
    return full_is, full_bs, final_df if unified_dfs else None

# ---------------------------
# RUN
# ---------------------------

print("🚀 Starting PriBank PDF financial data extraction...")
income_df, balance_df, final_df = main()

if final_df is not None:
    print("\n📊 FINAL UNIFIED DATAFRAME")
    print(f"Shape: {final_df.shape}")
    print("\nFirst 10 rows:")
    print(final_df.head(10))
    print(f"\n✅ Income Statement rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'INCOME_STATEMENT'])}")
    print(f"✅ Balance Sheet rows: {len(final_df[final_df['STATEMENT_TYPE'] == 'BALANCE_SHEET'])}")

📁 Directory ready: /Users/test/Desktop/Stuff/pri_financial_data/current/balance-sheet
📁 Directory ready: /Users/test/Desktop/Stuff/pri_financial_data/current/income-statement
📁 Output directory ready: /Users/test/Desktop/Stuff/pri_financial_data/output
🚀 Starting PriBank PDF financial data extraction...
📂 Found 8 PDF files for processing
   ✅ Processed: Bilanci-dhe-Treguesit-financiar-TM2-2025.pdf
   ✅ Processed: Bilanci-dhe-Treguesit-financiar-TM3-2024.pdf
   ✅ Processed: Bilanci-dhe-Treguesit-financiar-TM2-2024.pdf
   ✅ Processed: Bilanci-dhe-Treguesit-financiar-TM4-2025.pdf
   ✅ Processed: Bilanci-dhe-Treguesit-financiar-TM4-2024.pdf
   ✅ Processed: Bilanci-dhe-Treguesit-financiar-TM1-2025.pdf
   ✅ Processed: Bilanci-dhe-Treguesit-financiar-TM3-2025-2.pdf
   ✅ Processed: Treguesit-finanaciar-TM1-2023.pdf
💾 Excel report saved to: /Users/test/Desktop/Stuff/pri_financial_data/output/pri_financial_report_20260318_212300.xlsx
💾 Unified data saved to: /Users/test/Desktop/Stuff/pri_financi

In [6]:
final_df.to_csv('PRI.csv')